## Cell 1: Install Dependencies

In [38]:
%%capture
!pip install huggingface_hub httpx numpy scikit-learn PyMuPDF
%%capture
!pip install sentence-transformers groq httpx numpy scikit-learn PyMuPDF
print("✅ All dependencies installed")

## Cell 2: Configuration

In [55]:
import os
from getpass import getpass

# Free API key at: https://console.groq.com → API Keys → Create API Key
GROQ_TOKEN = getpass("Enter your Groq API key (gsk_...): ")
os.environ["GROQ_TOKEN"] = GROQ_TOKEN

CONFIG = {
    "model": "llama-3.3-70b-versatile",

    "papers_per_source": 5,
    "max_papers_to_process": 8,
    "chunk_size": 1200,
    "s2_api_key": "YOUR_KEY_HERE",
    "chunk_overlap": 200,
    "top_k_chunks": 10,
    "rank_weights": {"relevance": 0.50, "recency": 0.25, "citations": 0.25},
    "min_evidence_confidence": 0.4,
    "max_new_tokens": 1500,
    "temperature": 0.2,
    "embedding_model": "all-MiniLM-L6-v2",
    "cache_dir": "/content/research_cache",
    "cache_enabled": True,
}
os.makedirs(CONFIG["cache_dir"], exist_ok=True)
print(f"   Cache dir  : {CONFIG['cache_dir']}")
print(f"   Embed model: {CONFIG['embedding_model']}")

print(f"✅ Config loaded | model: {CONFIG['model']}")

Enter your Groq API key (gsk_...): ··········
   Cache dir  : /content/research_cache
   Embed model: all-MiniLM-L6-v2
✅ Config loaded | model: llama-3.3-70b-versatile


## Cell 3: Imports & LLM Helper

In [56]:
import json, re, time, math, hashlib, datetime
import urllib.parse, urllib.request
import xml.etree.ElementTree as ET
from dataclasses import dataclass
from typing import Optional
import sqlite3
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from sentence_transformers import SentenceTransformer
import httpx
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

try:
    import fitz
    PDF_SUPPORT = True
except ImportError:
    PDF_SUPPORT = False
    print("⚠️  PyMuPDF not found – abstracts only")

print(f"Loading embedding model '{CONFIG['embedding_model']}'...")
EMBEDDER = SentenceTransformer(CONFIG["embedding_model"])
print("✅ Embedding model ready")

CURRENT_YEAR = datetime.datetime.now().year

GROQ_ENDPOINT = "https://api.groq.com/openai/v1/chat/completions"
GROQ_HEADERS = {
    "Authorization": f"Bearer {GROQ_TOKEN}",
    "Content-Type": "application/json",
}


def log(emoji, agent, msg):
    ts = datetime.datetime.now().strftime("%H:%M:%S")
    print(f"{ts} {emoji} [{agent}]: {msg}")


def call_llm(system: str, user: str, max_tokens: int = None) -> str:
    payload = {
        "model": CONFIG["model"],
        "messages": [
            {"role": "system", "content": system},
            {"role": "user",   "content": user},
        ],
        "max_tokens": max_tokens or CONFIG["max_new_tokens"],
        "temperature": CONFIG["temperature"],
        "stream": False,
    }
    for attempt in range(3):
        try:
            r = httpx.post(GROQ_ENDPOINT, headers=GROQ_HEADERS,
                           json=payload, timeout=60)
            if r.status_code == 429:
                wait = int(r.headers.get("Retry-After", 20))
                log("⏳", "LLM", f"Rate limited – waiting {wait}s...")
                time.sleep(wait)
                continue
            r.raise_for_status()
            return r.json()["choices"][0]["message"]["content"].strip()
        except Exception as e:
            log("⚠️", "LLM", f"Attempt {attempt+1}/3 failed: {e}")
            time.sleep(5)
    log("❌", "LLM", "All attempts failed.")
    return ""


print("✅ LLM ready – Groq /", CONFIG["model"])

Loading embedding model 'all-MiniLM-L6-v2'...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Embedding model ready
✅ LLM ready – Groq / llama-3.3-70b-versatile


## Cell 3b: LLM Smoke Test

Run this **before** the full pipeline to confirm your token and model access work.

In [58]:
# ── 🔧 Quick LLM smoke-test ───────────────────────────────────────────────────
# Run this cell alone first to confirm the LLM is working before the full pipeline.
print("Testing LLM with a simple prompt...")
test_resp = call_llm(
    system="You are a helpful assistant. Reply in one short sentence.",
    user="What is 2 + 2?",
    max_tokens=50
)
if test_resp:
    print(f"✅ LLM working! Response: {test_resp}")
else:
    print("❌ LLM call failed. Check the errors above.")
    print("   Common fixes:")
    print("   1. Accept the Llama license at https://huggingface.co/meta-llama/Llama-3.3-70B-Instruct")
    print("   2. Make sure your token has 'Make calls to Inference Providers' permission")
    print("   3. Try the 8B model: change CONFIG['model'] to 'meta-llama/Llama-3.1-8B-Instruct'")


Testing LLM with a simple prompt...
✅ LLM working! Response: The answer is 4.


## Cell 4: Data Models

In [59]:
@dataclass
class Paper:
    title: str
    authors: list
    year: Optional[int]
    abstract: str
    url: str
    pdf_url: Optional[str] = None
    citation_count: int = 0
    source: str = "unknown"
    paper_id: str = ""
    relevance_score: float = 0.0
    recency_score: float = 0.0
    citation_score: float = 0.0
    final_score: float = 0.0

    def __post_init__(self):
        if not self.paper_id:
            self.paper_id = hashlib.md5(self.title.encode()).hexdigest()[:10]

    @property
    def short_ref(self):
        first = self.authors[0].split()[-1] if self.authors else "Unknown"
        return f"{first} et al. ({self.year or 'n.d.'})"


@dataclass
class Chunk:
    text: str
    paper: Paper
    chunk_index: int

    @property
    def citation_key(self):
        return self.paper.short_ref


@dataclass
class EvidenceItem:
    chunk: Chunk
    relevance_score: float
    verified: bool = False
    verification_note: str = ""


@dataclass
class ResearchAnswer:
    question: str
    rewritten_queries: list
    papers_found: int
    papers_processed: int
    evidence_items: list
    answer: str
    confidence_score: float
    sources: list


print("✅ Data models defined")

✅ Data models defined


## Cell 5: Agent 1 — Query Rewriter

In [60]:
def query_rewriter_agent(question):
    log("✏️", "QueryRewriter", f'Rewriting: "{question}"')

    system = """You are an expert academic search query optimizer.
Transform the user's research question into exactly 4 targeted search queries
for retrieving diverse, high-quality academic papers.

Query 1: exact technical terminology
Query 2: broader conceptual framing
Query 3: application / use-case angle
Query 4: recent advances / survey angle

Keep each query under 12 words.
Respond with ONLY a valid JSON array of 4 strings, nothing else.
Example: ["transformer attention mechanism", "sequence modelling deep learning", "NLP BERT applications industry", "survey large language models 2024"]"""

    raw = call_llm(system, question, max_tokens=200)
    match = re.search(r'\[.*?\]', raw, re.DOTALL)
    try:
        queries = json.loads(match.group()) if match else [question]
    except Exception:
        queries = [question]

    if question not in queries:
        queries.append(question)

    log("✏️", "QueryRewriter", f"Generated {len(queries)} queries:")
    for i, q in enumerate(queries, 1):
        print(f"   {i}. {q}")
    return queries


print("✅ Query Rewriter Agent ready")

✅ Query Rewriter Agent ready


## Cell 6: Agent 2 — Paper Search (S2 + arXiv + Crossref)

In [61]:
def search_semantic_scholar(query, limit=5):
    papers = []
    headers = {"x-api-key": CONFIG["s2_api_key"]} if CONFIG["s2_api_key"] else {}
    try:
        url = "https://api.semanticscholar.org/graph/v1/paper/search"
        params = {
            "query": query, "limit": limit,
            "fields": "title,authors,year,abstract,citationCount,openAccessPdf,url"
        }
        r = httpx.get(url, params=params, headers=headers, timeout=15)
        r.raise_for_status()
        for item in r.json().get("data", []):
            if not item.get("abstract"):
                continue
            pdf_url = (item.get("openAccessPdf") or {}).get("url")
            papers.append(Paper(
                title=item.get("title", "Untitled"),
                authors=[a.get("name", "") for a in item.get("authors", [])],
                year=item.get("year"),
                abstract=item["abstract"],
                url=item.get("url", ""),
                pdf_url=pdf_url,
                citation_count=item.get("citationCount") or 0,
                source="semantic_scholar",
                paper_id=item.get("paperId", ""),
            ))
    except Exception as e:
        log("⚠️", "Search", f"Semantic Scholar: {e}")
    return papers


def search_arxiv(query, limit=5):
    papers = []
    try:
        params = urllib.parse.urlencode({
            "search_query": f"all:{query}", "start": 0,
            "max_results": limit, "sortBy": "relevance"
        })
        with urllib.request.urlopen(
            f"http://export.arxiv.org/api/query?{params}", timeout=15
        ) as resp:
            root = ET.fromstring(resp.read())
        ns = {"atom": "http://www.w3.org/2005/Atom"}
        for entry in root.findall("atom:entry", ns):
            abstract = entry.findtext("atom:summary", "", ns).replace("\n", " ").strip()
            if not abstract:
                continue
            published = entry.findtext("atom:published", "", ns)
            arxiv_id = entry.findtext("atom:id", "", ns).split("/abs/")[-1]
            papers.append(Paper(
                title=entry.findtext("atom:title", "", ns).replace("\n", " ").strip(),
                authors=[a.findtext("atom:name", "", ns)
                         for a in entry.findall("atom:author", ns)],
                year=int(published[:4]) if published else None,
                abstract=abstract,
                url=f"https://arxiv.org/abs/{arxiv_id}",
                pdf_url=f"https://arxiv.org/pdf/{arxiv_id}.pdf",
                source="arxiv",
                paper_id=arxiv_id,
            ))
    except Exception as e:
        log("⚠️", "Search", f"arXiv: {e}")
    return papers


def search_crossref(query, limit=5):
    papers = []
    try:
        params = {
            "query": query, "rows": limit,
            "select": "title,author,published,abstract,DOI,is-referenced-by-count"
        }
        r = httpx.get("https://api.crossref.org/works", params=params, timeout=15,
                      headers={"User-Agent": "ResearchAgent/1.0"})
        r.raise_for_status()
        for item in r.json().get("message", {}).get("items", []):
            abstract = re.sub(r'<[^>]+>', '', item.get("abstract") or "").strip()
            title_list = item.get("title", [])
            if not title_list:
                continue
            authors = [f"{a.get('given','')} {a.get('family','')}".strip()
                       for a in item.get("author", [])]
            pub = item.get("published", {}).get("date-parts", [[None]])[0]
            doi = item.get("DOI", "")
            papers.append(Paper(
                title=title_list[0],
                authors=authors,
                year=pub[0] if pub else None,
                abstract=abstract,
                url=f"https://doi.org/{doi}" if doi else "",
                citation_count=item.get("is-referenced-by-count") or 0,
                source="crossref",
                paper_id=doi,
            ))
    except Exception as e:
        log("⚠️", "Search", f"Crossref: {e}")
    return papers


def paper_search_agent(queries):
    log("🔍", "PaperSearch", f"Searching {len(queries)} queries × 3 sources...")
    all_papers = []
    limit = CONFIG["papers_per_source"]
    for q in queries:
        log("🔍", "PaperSearch", f'  "{q}"')
        all_papers.extend(search_semantic_scholar(q, limit))
        time.sleep(2)
        all_papers.extend(search_arxiv(q, limit))
        all_papers.extend(search_crossref(q, limit))

    seen, unique = set(), []
    for p in all_papers:
        key = re.sub(r'[^a-z0-9]', '', p.title.lower())
        if key not in seen and p.abstract:
            seen.add(key)
            unique.append(p)

    log("🔍", "PaperSearch", f"{len(all_papers)} total → {len(unique)} unique")
    return unique


print("✅ Paper Search Agent ready")

✅ Paper Search Agent ready


## Cell 7: Paper Ranker

In [62]:
def score_relevance_with_llm(papers, question):
    """Batch-score abstract relevance with Llama."""
    system = """You are an academic relevance scorer.
Given a research question and paper abstracts, score each paper's relevance 0.0–1.0.
1.0=directly addresses it; 0.7=relevant but tangential; 0.4=loosely related; 0.1=unrelated.
Respond with ONLY a valid JSON array of numbers, one per paper, in input order.
Example for 3 papers: [0.9, 0.3, 0.7]
No explanation, no markdown, just the JSON array."""

    batch = f"Research question: {question}\n\nPapers:\n"
    for i, p in enumerate(papers):
        batch += f"[{i}] {p.title}\n{p.abstract[:250]}\n\n"

    raw = call_llm(system, batch, max_tokens=200)
    match = re.search(r'\[.*?\]', raw, re.DOTALL)
    try:
        scores = json.loads(match.group()) if match else []
    except Exception:
        scores = []
    for i, p in enumerate(papers):
        p.relevance_score = float(scores[i]) if i < len(scores) else 0.5
    return papers


def rank_papers(papers, question):
    log("📊", "Ranker", f"Scoring {len(papers)} papers...")
    score_relevance_with_llm(papers, question)

    papers = [p for p in papers if p.relevance_score >= 0.3]

    if not papers:
        log("⚠️", "Ranker", "No papers passed relevance threshold")
        return []

    max_cit = max((p.citation_count for p in papers), default=1) or 1

    for p in papers:
        p.citation_score = math.log1p(p.citation_count) / math.log1p(max_cit)
        age = max(0, CURRENT_YEAR - (p.year or CURRENT_YEAR))
        p.recency_score = math.exp(-0.15 * age)

    w = CONFIG["rank_weights"]
    for p in papers:
        p.final_score = (w["relevance"] * p.relevance_score
                       + w["recency"]   * p.recency_score
                       + w["citations"] * p.citation_score)

    ranked = sorted(papers, key=lambda p: p.final_score, reverse=True)
    top_n = CONFIG["max_papers_to_process"]
    log("📊", "Ranker", f"Top {top_n}:")
    for i, p in enumerate(ranked[:top_n], 1):
        print(f"   {i}. [{p.final_score:.2f}] {p.title[:65]}... ({p.year})")
    return ranked[:top_n]


print("✅ Paper Ranker ready")

✅ Paper Ranker ready


## Cell 8: PDF Downloader + Parser + Chunker

In [63]:
# ── Disk cache (SQLite) ───────────────────────────────────────────────────────
def _get_db():
    db_path = Path(CONFIG["cache_dir"]) / "paper_cache.db"
    conn = sqlite3.connect(str(db_path))
    conn.execute("""
        CREATE TABLE IF NOT EXISTS papers (
            paper_id  TEXT PRIMARY KEY,
            title     TEXT,
            chunks    TEXT,
            cached_at TEXT
        )
    """)
    conn.commit()
    return conn

def _cache_get(paper_id):
    if not CONFIG["cache_enabled"]:
        return None
    try:
        conn = _get_db()
        row = conn.execute(
            "SELECT chunks FROM papers WHERE paper_id = ?", (paper_id,)
        ).fetchone()
        conn.close()
        return json.loads(row[0]) if row else None
    except Exception:
        return None

def _cache_set(paper_id, title, chunk_texts):
    if not CONFIG["cache_enabled"]:
        return
    try:
        conn = _get_db()
        conn.execute(
            """INSERT OR REPLACE INTO papers (paper_id, title, chunks, cached_at)
               VALUES (?, ?, ?, ?)""",
            (paper_id, title, json.dumps(chunk_texts),
             datetime.datetime.now().isoformat())
        )
        conn.commit()
        conn.close()
    except Exception as e:
        log("⚠️", "Cache", f"Write failed: {e}")

def cache_stats():
    try:
        conn = _get_db()
        n = conn.execute("SELECT COUNT(*) FROM papers").fetchone()[0]
        conn.close()
        log("💾", "Cache", f"{n} papers cached at {CONFIG['cache_dir']}")
    except Exception:
        log("💾", "Cache", "Cache empty")

def download_pdf_text(paper):
    if not PDF_SUPPORT or not paper.pdf_url:
        return paper.abstract
    try:
        r = httpx.get(paper.pdf_url, timeout=20, follow_redirects=True)
        if r.status_code != 200 or "pdf" not in r.headers.get("content-type", ""):
            return paper.abstract
        doc = fitz.open(stream=r.content, filetype="pdf")
        text = "\n".join(page.get_text() for page in doc)
        text = re.sub(r'\n{3,}', '\n\n', text)
        text = re.sub(r'[^\x20-\x7E\n]', ' ', text)
        if len(text) > 500:
            return text
    except Exception:
        pass
    return paper.abstract

def chunk_text(text, paper):
    size, overlap = CONFIG["chunk_size"], CONFIG["chunk_overlap"]
    sentences = re.split(r'(?<=[.!?])\s+', text)
    chunks, current, idx = [], "", 0
    for sent in sentences:
        if len(current) + len(sent) > size and current:
            chunks.append(Chunk(text=current.strip(), paper=paper, chunk_index=idx))
            current = current[-overlap:] + " " + sent
            idx += 1
        else:
            current += " " + sent
    if current.strip():
        chunks.append(Chunk(text=current.strip(), paper=paper, chunk_index=idx))
    return chunks

def _process_paper(paper):
    cached = _cache_get(paper.paper_id)
    if cached is not None:
        log("💾", "Cache", f"HIT  '{paper.title[:50]}'  ({len(cached)} chunks)")
        return [Chunk(text=t, paper=paper, chunk_index=i)
                for i, t in enumerate(cached)]
    text = download_pdf_text(paper)
    chunks = chunk_text(text, paper)
    _cache_set(paper.paper_id, paper.title, [c.text for c in chunks])
    log("📄", "Ingestion", f"MISS '{paper.title[:50]}'  ({len(chunks)} chunks)")
    return chunks

def ingest_papers(papers):
    log("📄", "Ingestion", f"Processing {len(papers)} papers in parallel...")
    all_chunks = []
    with ThreadPoolExecutor(max_workers=4) as ex:
        futures = {ex.submit(_process_paper, p): p for p in papers}
        for future in as_completed(futures):
            try:
                all_chunks.extend(future.result())
            except Exception as e:
                log("⚠️", "Ingestion", f"Paper failed: {e}")
    log("📄", "Ingestion", f"Total: {len(all_chunks)} chunks")
    return all_chunks

print("✅ Parallel ingestion + disk cache ready")

✅ Parallel ingestion + disk cache ready


##  Cell 9: In-Memory Vector DB (TF-IDF)

In [64]:
class VectorDB:
    def __init__(self):
        self.chunks = []
        self._matrix = None

    def build_index(self, chunks):
        log("🗃️", "VectorDB", f"Embedding {len(chunks)} chunks...")
        self.chunks = chunks
        embeddings = EMBEDDER.encode(
            [c.text for c in chunks],
            batch_size=64,
            show_progress_bar=True,
            normalize_embeddings=True,
        )
        self._matrix = np.array(embeddings)
        log("🗃️", "VectorDB",
            f"Index: {self._matrix.shape[0]} × {self._matrix.shape[1]}")

    def retrieve(self, query, top_k=10):
        q_vec = EMBEDDER.encode([query], normalize_embeddings=True)
        scores = (q_vec @ self._matrix.T)[0]
        top_idxs = np.argsort(scores)[::-1][:top_k]
        return [(self.chunks[i], float(scores[i]))
                for i in top_idxs if scores[i] > 0.1]

print("✅ Semantic VectorDB ready (sentence-transformers)")

✅ Semantic VectorDB ready (sentence-transformers)


## Cell 10: Agent 3 — Evidence Verification

In [65]:
def evidence_verification_agent(question, retrieved):
    if not retrieved:
        return []
    log("🔎", "EvidenceVerifier", f"Verifying {len(retrieved)} passages...")

    passages = ""
    for i, (chunk, score) in enumerate(retrieved):
        passages += f"[{i}] score={score:.2f} | {chunk.citation_key}\n{chunk.text[:400]}\n\n"

    system = """You are a rigorous academic evidence verifier.
For each retrieved passage, evaluate:
1. Is it relevant to the research question? (true/false)
2. Confidence as supporting evidence (0.0–1.0)
3. One-sentence note explaining your decision

Respond with ONLY a valid JSON array, one object per passage in input order:
[{"relevant": true, "confidence": 0.85, "note": "Directly measures hallucination rates in LLMs"}, ...]

Be strict: reject passages that are off-topic, too vague, or only tangentially related.
No markdown, no explanation outside the JSON array."""

    raw = call_llm(system, f"Question: {question}\n\nPassages:\n{passages}", max_tokens=800)
    match = re.search(r'\[.*?\]', raw, re.DOTALL)
    try:
        verdicts = json.loads(match.group()) if match else []
    except Exception:
        verdicts = []

    items = []
    min_conf = CONFIG["min_evidence_confidence"]
    for i, (chunk, _) in enumerate(retrieved):
        v = verdicts[i] if i < len(verdicts) else {"relevant": True, "confidence": 0.5, "note": ""}
        conf = float(v.get("confidence", 0.5))
        if v.get("relevant", True) and conf >= min_conf:
            items.append(EvidenceItem(
                chunk=chunk, relevance_score=conf,
                verified=True, verification_note=v.get("note", "")
            ))

    log("🔎", "EvidenceVerifier", f"{len(items)}/{len(retrieved)} passages passed")
    return items


print("✅ Evidence Verification Agent ready")

✅ Evidence Verification Agent ready


## Cell 11: Agent 4 — Answer Generation

In [66]:
def answer_generation_agent(question, evidence_items):
    log("✍️", "AnswerGen", f"Generating from {len(evidence_items)} evidence items...")
    if not evidence_items:
        return "⚠️ Insufficient evidence found in the retrieved papers.", 0.0

    context = ""
    for ev in evidence_items:
        context += f"[{ev.chunk.citation_key}] (conf={ev.relevance_score:.2f})\n"
        context += ev.chunk.text[:600] + "\n\n"

    system = """You are a rigorous academic research synthesiser.

Write a comprehensive, well-structured answer that:
1. Directly answers the question in the first paragraph
2. Uses inline citations like [Author et al. (Year)] after every factual claim
3. Organises findings into clear sections where appropriate
4. Notes contradictions, caveats, or limitations in the evidence
5. Ends with a concise summary

After your answer, on a new line output ONLY this JSON (no markdown fences):
{"confidence": 0.85, "reasoning": "Strong consensus across 4 papers"}

Confidence guide:
  0.9-1.0 = strong consensus across multiple high-quality papers
  0.7-0.9 = good evidence with minor gaps
  0.5-0.7 = mixed or limited evidence
  0.3-0.5 = weak or indirect evidence
  <0.3    = essentially no reliable evidence

Write in clear academic prose."""

    raw = call_llm(system, f"Question: {question}\n\nEvidence:\n{context}", max_tokens=2000)

    conf_match = re.search(r'"confidence"\s*:\s*([0-9.]+)', raw)
    confidence = float(conf_match.group(1)) if conf_match else 0.6
    answer = re.sub(r'\{\s*"confidence".*?\}', '', raw, flags=re.DOTALL).strip()
    return answer, confidence


print("✅ Answer Generation Agent ready")

✅ Answer Generation Agent ready


## Cell 12: Orchestrator — Full Pipeline

In [67]:
def run_research_agent(question):
    t0 = time.time()
    print("\n" + "="*65)
    print("🔬  AUTONOMOUS RESEARCH AGENT  [Llama via Groq]")
    print("="*65)
    log("❓", "Orchestrator", f'Question: "{question}"')

    print("\n── Stage 1: Query Rewriting ──────────────────────────────────")
    queries = query_rewriter_agent(question)

    print("\n── Stage 2: Paper Search ─────────────────────────────────────")
    all_papers = paper_search_agent(queries)
    if not all_papers:
        log("❌", "Orchestrator", "No papers found. Try rephrasing the question.")
        return ResearchAnswer(question, queries, 0, 0, [], "No papers found.", 0.0, [])

    print("\n── Stage 3: Ranking ──────────────────────────────────────────")
    ranked = rank_papers(all_papers, question)

    cache_stats()
    print("\n── Stage 4: Ingestion ─────────────────────────────────────")

    print("\n── Stage 4: Ingestion ────────────────────────────────────────")
    chunks = ingest_papers(ranked)

    print("\n── Stage 5: Vector Indexing ──────────────────────────────────")
    db = VectorDB()
    db.build_index(chunks)

    print("\n── Stage 6: Retrieval ────────────────────────────────────────")
    retrieved = db.retrieve(question, top_k=CONFIG["top_k_chunks"])
    log("🎯", "Retrieval", f"{len(retrieved)} candidate passages")

    print("\n── Stage 7: Evidence Verification ───────────────────────────")
    evidence = evidence_verification_agent(question, retrieved)

    print("\n── Stage 8: Answer Generation ────────────────────────────────")
    answer, confidence = answer_generation_agent(question, evidence)

    log("⏱️", "Orchestrator", f"Done in {time.time()-t0:.1f}s")

    seen, sources = set(), []
    for ev in evidence:
        pid = ev.chunk.paper.paper_id
        if pid not in seen:
            seen.add(pid)
            sources.append(ev.chunk.paper)

    return ResearchAnswer(question, queries, len(all_papers), len(ranked),
                          evidence, answer, confidence, sources)


print("✅ Orchestrator ready")

✅ Orchestrator ready


## Cell 13: Output Formatter

In [68]:
def display_result(r):
    D = "═" * 68
    filled = int(r.confidence_score * 20)
    bar = "█" * filled + "░" * (20 - filled)
    label = ("🟢 High"   if r.confidence_score >= 0.75 else
             "🟡 Medium" if r.confidence_score >= 0.50 else "🔴 Low")

    print(f"\n{D}")
    print("📋  RESEARCH ANSWER")
    print(D)
    print(f"❓  {r.question}")
    print(f"📚  Papers: {r.papers_found} found → {r.papers_processed} processed")
    print(f"🔬  Evidence passages used: {len(r.evidence_items)}")
    print(f"🎯  Confidence: [{bar}] {r.confidence_score:.0%}  {label}")

    print(f"\n{D}")
    print("📝  ANSWER")
    print(D)
    print(r.answer)

    print(f"\n{D}")
    print("📖  REFERENCES")
    print(D)
    for i, p in enumerate(r.sources, 1):
        au = ", ".join(p.authors[:3]) + (" et al." if len(p.authors) > 3 else "")
        print(f"[{i}] {au} ({p.year or 'n.d.'}). {p.title}")
        if p.url:
            print(f"     🔗 {p.url}")
    print(f"\n{D}\n")


def export_markdown(r, path="research_answer.md"):
    lines = [f"# Research Answer\n",
             f"**Question:** {r.question}\n",
             f"**Model:** {CONFIG['model']}\n",
             f"**Confidence:** {r.confidence_score:.0%}\n",
             f"**Papers processed:** {r.papers_processed}\n\n---\n## Answer\n",
             r.answer, "\n\n---\n## References\n"]
    for i, p in enumerate(r.sources, 1):
        au = ", ".join(p.authors[:3]) + (" et al." if len(p.authors) > 3 else "")
        lines.append(f"{i}. {au} ({p.year or 'n.d.'}). *{p.title}*. {p.url}\n")
    with open(path, "w") as f:
        f.write("\n".join(lines))
    print(f"✅ Saved to {path}")


def inspect_evidence(r, n=5):
    print(f"\n{'─'*60}\n🔍 TOP {n} EVIDENCE PASSAGES\n{'─'*60}")
    top = sorted(r.evidence_items, key=lambda e: e.relevance_score, reverse=True)[:n]
    for i, ev in enumerate(top, 1):
        print(f"\n[{i}] {ev.chunk.citation_key}  conf={ev.relevance_score:.2f}")
        print(f"    {ev.verification_note}")
        print(f"    {ev.chunk.text[:250]}...")


print("✅ Output formatter ready")

✅ Output formatter ready


## Cell 14: Run the Agent

Edit `QUESTION` then **Run All** (or run cells 1–13 first, then this cell).

In [69]:
# ════════════════════════════════════════════════════════
#  RESEARCH QUESTION HERE
# ════════════════════════════════════════════════════════
QUESTION = "What are the most effective methods for reducing hallucinations in large language models?"
# ════════════════════════════════════════════════════════

result = run_research_agent(QUESTION)
display_result(result)

# Optional
export_markdown(result)
inspect_evidence(result, n=5)


🔬  AUTONOMOUS RESEARCH AGENT  [Llama via Groq]
18:53:22 ❓ [Orchestrator]: Question: "What are the most effective methods for reducing hallucinations in large language models?"

── Stage 1: Query Rewriting ──────────────────────────────────
18:53:22 ✏️ [QueryRewriter]: Rewriting: "What are the most effective methods for reducing hallucinations in large language models?"
18:53:22 ✏️ [QueryRewriter]: Generated 5 queries:
   1. hallucination mitigation techniques
   2. language model factual accuracy
   3. reducing hallucinations in chatbots
   4. recent advances in hallucination control
   5. What are the most effective methods for reducing hallucinations in large language models?

── Stage 2: Paper Search ─────────────────────────────────────
18:53:22 🔍 [PaperSearch]: Searching 5 queries × 3 sources...
18:53:22 🔍 [PaperSearch]:   "hallucination mitigation techniques"
18:53:26 🔍 [PaperSearch]:   "language model factual accuracy"
18:53:29 🔍 [PaperSearch]:   "reducing hallucinations in cha

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

18:54:08 🗃️ [VectorDB]: Index: 57 × 384

── Stage 6: Retrieval ────────────────────────────────────────
18:54:08 🎯 [Retrieval]: 10 candidate passages

── Stage 7: Evidence Verification ───────────────────────────
18:54:08 🔎 [EvidenceVerifier]: Verifying 10 passages...
18:54:10 🔎 [EvidenceVerifier]: 8/10 passages passed

── Stage 8: Answer Generation ────────────────────────────────
18:54:10 ✍️ [AnswerGen]: Generating from 8 evidence items...
18:54:11 ⏱️ [Orchestrator]: Done in 49.7s

════════════════════════════════════════════════════════════════════
📋  RESEARCH ANSWER
════════════════════════════════════════════════════════════════════
❓  What are the most effective methods for reducing hallucinations in large language models?
📚  Papers: 45 found → 8 processed
🔬  Evidence passages used: 8
🎯  Confidence: [█████████████████░░░] 85%  🟢 High

════════════════════════════════════════════════════════════════════
📝  ANSWER
════════════════════════════════════════════════════════════════════